# Hour 3 · Divination as decision trees

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alexsosn/ugarit-dh-workshop/blob/main/notebooks/3c_divination_trees.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/alexsosn/ugarit-dh-workshop/main?labpath=notebooks%2F3c_divination_trees.ipynb)


**Goal:** formalize an omen text's *if → then* logic as data and visualize it as a tree — then discuss where LLMs help and where they mislead.

We use a real Ugaritic **birth-omen** text and a hand-built decision tree of it (`data/omens/`).

**Needs:** `networkx`.

*Reading:* `docs/07-divination.md`.

## 0. Setup


In [1]:
# === SETUP — run me first (works in Google Colab, Binder, and locally) ===
import os, sys, subprocess

if "google.colab" in sys.modules:                      # we're on Colab
    REPO_URL = "https://github.com/alexsosn/ugarit-dh-workshop.git"
    REPO_DIR = "/content/ugarit-dh-workshop"
    if not os.path.isdir(REPO_DIR):                    # clone the repo once
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(os.path.join(REPO_DIR, "notebooks"))      # work from notebooks/
    # Colab already ships numpy/pandas/scikit-learn/matplotlib/plotly/networkx;
    # only umap-learn is usually missing. Install just that (fast).
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "umap-learn"], check=False)

# make data/loader.py importable (we run from the notebooks/ folder)
sys.path.insert(0, os.path.abspath(".."))
from data.loader import load_omen_text, load_texts

texts = load_texts()     # 1st Colab run downloads the CUC JSONL from HuggingFace, then caches it
print(f"Loaded {len(texts)} tablets — genre of the first one: {texts[0]['genre']}")

[loader] Loaded 278 CUC tablets, 25477 word tokens (source: AlexWalhai/cuc JSONL cache, licence: CC BY-NC 4.0).
Loaded 278 tablets — genre of the first one: myth


## 1. The omen text (English working translation)

The sample is Dennis Pardee's translation of **RS 24.247+** (KTU 1.103+1.145), the Ugaritic *Malformed Animal Fetuses* omens (Pardee, *Ritual and Cult at Ugarit*, 2002). Note the conditional pattern: *if the newborn lacks or shows feature X → consequence for land, king, or cattle*.

In [2]:
txt = load_omen_text()
print(txt[:900])


 As for the ewes of the sheep/goats, [when t]hey give birth:

 If it is a stone, many will fall in the land. 
 
 If it is a piece of wood, behold [ ]ôYû <ATR YLD, its cattle will be destroyed. 
 If the fetus is smooth, without h[air?], there will be []...in the land. 
 And if i[t has no ], the land will perish. 
 [ ] there will be famine in the land. 
 [ ] nor nostrils, the land [ ;] ditto. 
 [And] if it has no [ ], the king will sieze the lan[d of his enemy and?] the weapon of the king will lay the land low. 
 [ ] [ ] cattle [ will peri]sh.? 
 And if it has no [left] thigh, the king will [ ] his enemy. 
 And if there is no lower left leg, the king [will ] his enemy. 
 And if there is a horn of flesh [in] its lef[t te]mple, [ ]. 
 If it has no spleen [ ] [ ;] di[tto;] 
 the king will not obtain offspring. 
 [And] if it has no testicles, the (seed-)gra[in ].5 
 And if the middle part of 


## 2. The same logic, formalized as a tree (nested JSON)

In [3]:
from data.loader import load_omen_tree
import json
tree = load_omen_tree()
print(json.dumps(tree, indent=2, ensure_ascii=False)[:700], "...")

{
  "_source": "Dennis Pardee, Ritual and Cult at Ugarit (WAW 10, 2002), Text 42 = RS 24.247+ (KTU 1.103+1.145), Malformed Animal Fetuses. English translation: Pardee 2002.",
  "omen": {
    "sheep birth": {
      "missing body part": {
        "[body part] 1": "the land will perish",
        "[body part] 2": "the king will sieze the lan[d of his enemy and?] the weapon of the king will lay the land low",
        "nostrils": "[...]",
        "[...] and nostrils": "the land [...]; there will be famine in the land",
        "ḤRṢP in [its?] K[...]": "[...]",
        "thigh": {
          "left": "the king will [...] his enemy",
          "right": "[...]"
        },
        "leg": {
          "low ...


## 3. Turn the nested dict into a graph

In [4]:
import networkx as nx
G = nx.DiGraph()
counter = [0]
def add(node_label, subtree, parent=None):
    nid = counter[0]; counter[0]+=1
    G.add_node(nid, label=str(node_label), leaf=not isinstance(subtree, dict))
    if parent is not None: G.add_edge(parent, nid)
    if isinstance(subtree, dict):
        for k, v in subtree.items():
            add(k, v, nid)
    else:
        lid = counter[0]; counter[0]+=1
        G.add_node(lid, label="→ "+str(subtree), leaf=True)
        G.add_edge(nid, lid)
# Draw ONLY the sheep-birth omens here; the (separate) lunar tablet (RIH 78/14) is in §6.
add("sheep birth (RS 24.247+)", tree["omen"]["sheep birth"])
print("nodes:", G.number_of_nodes())

nodes: 77


## 4. Draw the sheep-birth decision tree

This is the **sheep-birth** tablet — **RS 24.247+** (KTU 1.103+1.145), Pardee's *Malformed Animal Fetuses*. Its data file also carries a short set of **lunar** omens, but those are a *different* tablet (RIH 78/14) — we draw them on their own in §6, not mixed into this tree.

Each **leaf is tinted by its outcome** — green = favorable, red = unfavorable, grey = neutral, pale = broken/missing — using the very same keyword rule we *quantify* in §8. (Branch nodes are amber.)

In [5]:
import collections, textwrap, re
import plotly.graph_objects as go

# --- Outcome polarity: a transparent keyword rule. The SAME rule is MEASURED in §8;
#     here it tints each leaf (green favorable · red unfavorable · grey neutral · pale broken). ---
BAD = ["perish","die","death","destroy","devastat","famine","scatter","fall","defeat","plague",
       "depopulat","misery","revolt","rebel","abandon","dwindle","lost","poverty","dire","mourn",
       "drought","kill","devour","consume", "consumed", "tread","captured","be taken","hard times","lay low",
       "flogged","not obtain","put down","turn against","dry up","deluge","cataclysm","starvation",
       "sick","epidemic","go mad","grow nervous","cut off","demolish","war","deranged","abase",
       "shorten","occupy the throne","seize the throne", "hardship", "days are over", "criminals",
       "will be affected"]
GOOD = ["prosper","abundance","safe","increase","thrive","peace","populous","return home","bounty",
        "expand","strengthen","more powerful","all-powerful","raised","rule","recover", "rain", 
        "the king will lay the land low"]

def polarity(text):
    """favorable / unfavorable / neutral / broken-missing — a rough, transparent heuristic."""
    t = text.lower()
    if text.strip() in ("[...]", "→ [...]", "") or (text.count("[") and sum(c.isalpha() for c in text) < 6):
        return "broken/missing"
    foe = "enemy" in t or "foe" in t
    win = foe and any(w in t for w in ["destroy","devastat","seize","lay low",
                                       "encounter no equal","make peace","conquer"])
    enemy_attacks = bool(re.search(r"(enemy|gutian|elam) (will|army)", t)) and not win
    if win and not enemy_attacks: return "favorable"         # OUR king beats the enemy
    if enemy_attacks:             return "unfavorable"       # the enemy beats us
    bad  = sum(w in t for w in BAD)
    good = sum(w in t for w in GOOD)
    return "favorable" if good > bad else "unfavorable" if bad > good else "neutral"

POLARITY_COLOR = {"favorable": "#7fb285", "unfavorable": "#e07a5f",
                  "neutral": "#c7cdd4", "broken/missing": "#ece7d9"}

def wrap_label(text, width=50):
    return "<br>".join(textwrap.wrap(text, width=width)) or text

def tidy_tree_pos(G, root=0):
    """Tidy layout: x = depth, parents centred on children; each leaf row is spaced by its
    line-count so full (wrapped) outcome text never overlaps."""
    depth = {root: 0}
    q = collections.deque([root])
    while q:
        u = q.popleft()
        for v in G.successors(u):
            depth[v] = depth[u] + 1
            q.append(v)
    cur = [0.0]; pos = {}
    def place(u):
        kids = list(G.successors(u))
        if not kids:
            lines = len(wrap_label(G.nodes[u]["label"]).split("<br>"))
            y = cur[0] + (lines - 1) / 2.0
            cur[0] += lines + 0.5
        else:
            y = sum(place(k) for k in kids) / len(kids)
        pos[u] = (depth[u], -y)
        return y
    place(root)
    return pos, cur[0]

def plot_interactive_tree(G, title, root=0):
    pos, span = tidy_tree_pos(G, root=root)
    nodes = list(G.nodes())
    is_leaf = [G.out_degree(n) == 0 for n in nodes]
    max_depth = max(int(x) for x, _ in pos.values())

    def colour(n, lf):
        return POLARITY_COLOR[polarity(G.nodes[n]["label"].lstrip("→ ").strip())] if lf else "#f8dfaa"

    edge_x, edge_y = [], []
    for u, v in G.edges():
        xu, yu = pos[u]; xv, yv = pos[v]
        edge_x += [xu, xv, None]; edge_y += [yu, yv, None]

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=edge_x, y=edge_y, mode="lines",
        line=dict(color="rgba(150,150,150,0.45)", width=1), hoverinfo="skip", showlegend=False))
    fig.add_trace(go.Scatter(
        x=[pos[n][0] for n in nodes], y=[pos[n][1] for n in nodes], mode="markers+text",
        text=[wrap_label(G.nodes[n]["label"]) if lf else G.nodes[n]["label"]
              for n, lf in zip(nodes, is_leaf)],
        textposition="middle right", textfont=dict(size=10, color="#1f2937"),
        marker=dict(size=12, color=[colour(n, lf) for n, lf in zip(nodes, is_leaf)],
                    line=dict(color="#6b7280", width=1)),
        hovertext=[G.nodes[n]["label"] for n in nodes],
        hovertemplate="%{hovertext}<extra></extra>", showlegend=False))
    legend = [("condition", "#f8dfaa"), ("favorable", POLARITY_COLOR["favorable"]),
              ("unfavorable", POLARITY_COLOR["unfavorable"]), ("neutral", POLARITY_COLOR["neutral"]),
              ("broken/missing", POLARITY_COLOR["broken/missing"])]
    for name, col in legend:
        fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", name=name,
            marker=dict(size=11, color=col, line=dict(color="#6b7280", width=1))))

    fig.update_layout(title=title,
        width=max(1000, (max_depth + 1) * 300 + 430),
        height=max(440, span * 19 + 120),
        margin=dict(l=20, r=410, t=70, b=20),
        legend=dict(orientation="h", y=1.03, x=0, font=dict(size=11)),
        paper_bgcolor="white", plot_bgcolor="white",
        xaxis=dict(visible=False, range=[-0.4, max_depth + 1.6]),
        yaxis=dict(visible=False), dragmode="pan")
    fig.show()

plot_interactive_tree(G, "Sheep-birth omen series — RS 24.247+ (KTU 1.103+1.145)")

## 5. Birth omens across two worlds — teratomancy

Birth-anomaly omens (*teratomancy*) were a shared Near-Eastern science. The Ugaritic and Babylonian versions are **two branches of one genre**, not copies of each other: Pardee stresses that the Ugaritic texts correspond to *no* known Mesopotamian tablet and read more like a *reasoned overview* than the sprawling Babylonian compendia.

Strikingly, **both** cultures split birth omens into **animal** and **human**:

| | Ugaritic (Pardee) | Babylonian (George) |
|---|---|---|
| **animal** births | RS 24.247+ — the §4 sheep tree | no. 12 — *miscarried foetus* |
| **human** births | RS 24.302 — a tiny fragment\* | no. 35 — *Šumma izbu* Tablet I |

We can tree three of these four; the Ugaritic human text is too broken. Same grammar throughout (**protasis → apodosis**), same fixation on the fate of **king and land**.

> *šumma sinništu nēša ūlid* → "If a woman gives birth to a lion: that town will be taken, its king will be captured." — *Šumma izbu* I 5 *(transliteration normalized from George 2013)*

\* RS 24.302 (KTU 1.140) preserves little beyond the opening *"when a woman gives birth …"* — but its very existence shows Ugarit, like Babylon, distinguished human from animal teratomancy.

In [6]:
# Pick up loader.py edits without a kernel restart (avoids a stale-import error).
import importlib, data.loader
importlib.reload(data.loader)
from data.loader import load_babylonian_izbu_tree, load_babylonian_foetus_tree

bab  = load_babylonian_izbu_tree()      # No. 35 — Šumma izbu I (human births)
foet = load_babylonian_foetus_tree()    # No. 12 — Old Babylonian miscarried-foetus omens

# Reusable: nested dict → graph (same recipe as the Ugaritic tree in cells 3–4).
def build_omen_graph(subtree, root_label):
    g = nx.DiGraph(); ctr = [0]
    def add(node_label, st, parent=None):
        nid = ctr[0]; ctr[0] += 1
        g.add_node(nid, label=str(node_label), leaf=not isinstance(st, dict))
        if parent is not None:
            g.add_edge(parent, nid)
        if isinstance(st, dict):
            for k, v in st.items():
                add(k, v, nid)
        else:
            lid = ctr[0]; ctr[0] += 1
            g.add_node(lid, label="→ " + str(st), leaf=True)
            g.add_edge(nid, lid)
    add(root_label, subtree)
    return g

print("No. 12 — miscarried-foetus omens, read by body feature:")
for feat, apo in foet["omen"]["horns"].items():
    print(f"  horns: {feat:24s} → {apo}")

print("\nNo. 35 — Šumma izbu I, human-birth omens:")
for who, apo in list(bab["omen"]["gives birth to an animal"].items())[:5]:
    print(f"  gives birth to a {who.split(' (')[0]:9s} → {apo}")

No. 12 — miscarried-foetus omens, read by body feature:
  horns: two on right, one on left → symbol of Sargon, who had no equal
  horns: two on left, one on right → enemy will hem you in and take away your territory
  horns: protruding               → there will be an all-powerful king in the land; alternative: the dynasty of the king and his sons is at an end

No. 35 — Šumma izbu I, human-birth omens:
  gives birth to a lion      → that town will be taken, its king will be captured
  gives birth to a wolf      → the people's minds will become deranged
  gives birth to a dog       → the head of the household will die and that household will be scattered; the people's minds will become deranged; plague will devour
  gives birth to a pig       → a woman will occupy the throne
  gives birth to a ox        → there will be an all-powerful king in the land


In [7]:
# No. 12 — the miscarried-foetus tree: the DIRECT parallel to the Ugaritic sheep omens
Gf = build_omen_graph(foet["omen"], "miscarried foetus")
plot_interactive_tree(Gf, "Babylonian teratomancy — miscarried-foetus omens (George no. 12)")

In [8]:
# No. 35 — the vivid Šumma izbu Tablet I human-birth tree
Gb = build_omen_graph(bab["omen"], "Šumma izbu I")
plot_interactive_tree(Gb, "Babylonian Šumma izbu Tablet I — human-birth omens (George no. 35)")

### Three birth-omen trees, two traditions

|  | Bab. *miscarried foetus* (no. 12) | Bab. *Šumma izbu* I (no. 35) | Ugaritic **RS 24.247+** |
|---|---|---|---|
| Birth of… | a malformed **animal** foetus | a **human** baby | a **sheep / goat** |
| Read by… | **body feature** (horns, eyes, hooves) | **species / shape** of the newborn | missing or odd **body part** |
| Form | protasis → apodosis | protasis → apodosis | protasis → apodosis |
| Outcome | king, land, dynasty, famine | king, throne, land, dynasty | king, land, cattle, enemy |
| Date | Old Babylonian (early 2nd mill.) | Standard Babylonian (1st mill.) | Late Bronze Age Ugarit |
| Source | George 2013, no. 12 | George 2013, no. 35 | Pardee 2002, text 42 |

The **foetus** omens (no. 12) are the closest kin to the Ugaritic text — both inspect a deformed **animal** newborn, feature by feature. The **human** *Šumma izbu* I (no. 35) is the same tradition's vivid first chapter, and Ugarit has its own (fragmentary) human counterpart (RS 24.302). All three trees run the identical *if → then* engine, fixated on the fate of king and land.

But recall the caution (`docs/07-divination.md`): the Babylonian series are **systematic** but not **empirical** (George — entries by analogy and wordplay, even impossible births), while the Ugaritic texts read as a **reasoned overview of the possibilities** (Pardee). The trees capture the *structure* of the reasoning, not a log of things observed — exactly the nuance an LLM will smooth over.

## 6. The same logic in the sky — celestial omens

Birth omens are only one branch. Ugarit also read the **heavens**, in a *separate* tablet — **RIH 78/14** (KTU 1.163), Pardee's *Lunar Omens* (the colour and newness of the moon, a falling star). Babylon did the same on a far grander scale: *Enūma Anu Enlil*, in which a **lunar eclipse** is decoded by its **watch**, the **direction** it clears, and above all the **calendar date** — every month, every key day, its own prediction. We draw the small Ugaritic lunar tablet as its own tree, then the Babylonian eclipse calendar as a **grid** (its natural shape):

In [9]:
import pandas as pd
from data.loader import load_babylonian_celestial_tree, load_ugaritic_lunar_tree

# Ugaritic lunar omens — RIH 78/14, a SEPARATE tablet, drawn as its own little tree
ug_cel = load_ugaritic_lunar_tree()["omen"]
Gc = build_omen_graph(ug_cel, "Ugaritic lunar omens (RIH 78/14)")
plot_interactive_tree(Gc, "Ugaritic lunar omens — RIH 78/14 (a separate tablet)")

# Babylonian lunar-eclipse — the 'general' rules, then the systematic month × day grid
cel = load_babylonian_celestial_tree()
ecl = cel["omen"]["celestial omens"]["lunar eclipse"]
print("Babylonian lunar-eclipse, general rules (a sample):")
for k, v in list(ecl["general"].items())[:4]:
    print(f"  eclipse {k:28s} → {v}")

grid = pd.DataFrame(ecl["month-specific"]).T
grid = grid.reindex(columns=sorted(grid.columns, key=lambda c: int(c.split()[1]))).fillna("—")
print(f"\nBabylonian eclipse calendar: {grid.shape[0]} months × {grid.shape[1]} key days "
      f"— every cell is its own omen:")
grid

Babylonian lunar-eclipse, general rules (a sample):
  eclipse evening watch                → signifies a fatal epidemic
  eclipse middle watch                 → signifies the decline of commerce
  eclipse morning watch                → signifies the end of a dynasty (or recovery from sickness)
  eclipse right-hand side reversed     → deluge will occur everywhere

Babylonian eclipse calendar: 11 months × 6 key days — every cell is its own omen:


,day 14,day 15,day 16,day 19,day 20,day 21
Nisannu,downfall of the king of Akkad; city and people...,famine; people will trade their infant childre...,—,redness of moon is orange-hued; abundance for ...,king will declare war on king,commerce will dwindle
Ayyaru,all-powerful king will die; land will perish,king and people are safe; war declared; shortf...,king of Amurru will die; his people will perish,—,—,—
Simanu,famous king will die; a prince will seize the ...,king of Amurru’s servants will kill him in revolt,king will fall in battle; watercourse will dry up,—,—,—
Dumuzi,inundation reduces barley; large people seek r...,rains removed; starvation,evil in the land; bounty lost,—,—,—
Abu,famine; king of Amurru will die,Nergal will devour; fatalities in the land,ewes will not bring lambs to term,—,—,—
Elul,king’s brother will seize throne in a revolt,king’s son will kill father and seize the throne,king of the upper land will enter deserted land,—,—,—
Tashritu,attack by Elam on my land; agriculture will th...,locust swarm will arise in spring and strike c...,king will die; dynasty over; land will be at war,—,—,—
Kislimu,attack by Gutian army; people will perish,enemy will demolish fortifications,king’s auxiliaries and troops will turn hostil...,—,fog and constant rumbles of thunder,civil war
Tebetu,"Adad will devastate men, livestock, and wild a...",rain removed; land’s crops lost,king without claim will seize throne; war,—,a people that experienced deportation will ret...,king of a lost people will return home
Addaru,dogs will go mad and bite people,lions will rampage and cut off traffic,an undefeated land will be defeated,—,defeat of Ur,Gutian army will conquer the land of Akkad


The Ugaritic sky omens are far **smaller** than the Babylonian system — a few lines beside a calendar that assigns a fate to an eclipse on *every* key day of *every* month. That captures George's point about the Babylonian series: "the outcome of cumulative attempts to embrace the entire universe in a system of reciprocal inferences," a would-be **theory of everything**. Yet Pardee cautions that the Ugaritic text is not a Babylonian *import* but an independent **Western** version of the genre. And birth + sky are not all: Ugarit's divinatory repertoire runs to **four** manuals (Pardee texts 42–45) — animal *and* human teratomancy, lunar omens, and **dream** omens (RS 18.041). See `docs/07`.

## 7. A third technique — dream omens

Birth and sky were not the only signs Ugarit read. A fragmentary tablet — **RS 18.041**, Pardee's *Dream Omens* (text 45) — opens *"Document of dreams"* (*spr ḥlmm*) and lists the **meaning of things seen in a dream**: animals, tools, people. Most interpretations are broken away, but the *items* are revealing — the gods' own animals (the **young bull of Baʼlu**, the **horse of ʿAṭtartu**) sit beside everyday household objects (an axe, cups, sandals). Same divinatory grammar — *observed item → meaning* — applied to oneiromancy.

In [10]:
from data.loader import load_ugaritic_dream_tree

dream = load_ugaritic_dream_tree()
print(dream["_source"], "\n")
for item, meaning in dream["omen"]["cattle"].items():
    print(f"  dream of: {item:34s} → {meaning}")

Gd = build_omen_graph(dream["omen"], "Ugaritic dream omens (RS 18.041)")
plot_interactive_tree(Gd, "Ugaritic dream omens — RS 18.041 (oneiromancy)")

Dennis Pardee, Ritual and Cult at Ugarit (WAW 10, 2002), Text 45 = RS 18.041, Dream Omens. English translation: Pardee 2002. Heavily fragmentary; '[...]' = break. 

  dream of: year-old bull                      → [...]
  dream of: mature bull (two years)            → the word (interpretation?) [...]
  dream of: the young bull of Baʼlu            → [...]
  dream of: heifer about to be slaughtered     → [...]


## 8. Pass 1 — annotate the outcomes with keywords

Two analytical questions about the outcomes: *what mood?* (favorable / unfavorable) and *what concern?* (king, land, enemy…). We answer them in **two passes**. **Pass 1** is a fast, transparent **keyword** annotation — one **polarity** label and a multi-label list of **concern tags** per outcome. Rough on purpose; §9 lets an LLM fix it.

(Concerns are **tags**, not exclusive classes: *"famine; the king of Amurru will die"* → **famine + king + death/disease**. The taxonomy splits **death/disease** from natural **disaster**, and folds war into **enemy/war**.)

In [11]:
import collections
import pandas as pd
from data.loader import (load_omen_tree, load_ugaritic_lunar_tree, load_ugaritic_dream_tree,
                         load_babylonian_foetus_tree, load_babylonian_izbu_tree,
                         load_babylonian_celestial_tree)

# multi-label concern taxonomy (death/disease ≠ natural disaster; war → enemy/war)
TAGS = {
    "king":              ["king","ruler","prince","our lord","royal","reign"],
    "land":              ["land","country","territory"," city","town","akkad","amurru","subartu","sumer","elam"],
    "enemy/war":         ["enemy","foe","war","army","troops","battle","attack","archers","gutian","hostile","invade","revolt","conquer"],
    "famine/harvest":    ["famine","grain","barley","crop","harvest","starvation","seed","bounty","abundance","food","locust","inundation"],
    "dynasty/succession":["dynasty","throne","succession","usurp","rebel","occupy the throne","seize the throne","palace"],
    "people":            ["people","personnel","populous","population","servant","infant","woman","women"],
    "cattle/livestock":  ["cattle","livestock","sheep","lamb","goat","flock","progeny"],
    "death/disease":     ["die","death","perish","kill","plague","epidemic","fatal","mourn"],
    "disaster":          ["deluge","flood","cataclysm","drought","earthquake","storm","devour"],
}
def tags_of(text):                                   # multi-label keyword tagging
    t = text.lower()
    return [tag for tag, cues in TAGS.items() if any(c in t for c in cues)]

def apodoses(node):
    if isinstance(node, dict):
        out = []
        for v in node.values(): out += apodoses(v)
        return out
    return [str(node)]

SETS = {
    "Ug. animal birth":  apodoses(load_omen_tree()["omen"]["sheep birth"]),
    "Ug. lunar":         apodoses(load_ugaritic_lunar_tree()["omen"]),
    "Ug. dream":         apodoses(load_ugaritic_dream_tree()["omen"]),
    "Bab. foetus":       apodoses(load_babylonian_foetus_tree()["omen"]),
    "Bab. izbu (human)": apodoses(load_babylonian_izbu_tree()["omen"]),
    "Bab. eclipse":      apodoses(load_babylonian_celestial_tree()["omen"]),
}
# Pass 1: polarity (the same rule that tints the trees, §4) + concern tags, for EVERY outcome
pass1 = {s: [{"outcome": a, "polarity": polarity(a), "tags": tags_of(a)} for a in apos]
         for s, apos in SETS.items()}
print(sum(len(v) for v in pass1.values()), "outcomes annotated "
      "(== committed data/omens/annotations_pass1.json)\n")
print(json.dumps(pass1["Ug. animal birth"][:3], ensure_ascii=False, indent=1))

def poltable(ann):                                   # 5 scorable sets (dream is all-broken, see §9)
    rows = {}
    for s, recs in ann.items():
        if s == "Ug. dream": continue
        c = collections.Counter(r["polarity"] for r in recs)
        tot = sum(c.values()); read = tot - c["broken/missing"]
        rows[s] = {"total": tot, "broken": c["broken/missing"], "unfav": c["unfavorable"],
                   "neut": c["neutral"], "fav": c["favorable"],
                   "% bad": round(100*c["unfavorable"]/read) if read else 0,
                   "% good": round(100*c["favorable"]/read) if read else 0}
    return pd.DataFrame(rows).T

tbl1 = poltable(pass1)
print("\nPASS 1 — polarity:\n", tbl1.to_string())

175 outcomes annotated (== committed data/omens/annotations_pass1.json)

[
 {
  "outcome": "the land will perish",
  "polarity": "unfavorable",
  "tags": [
   "land",
   "death/disease"
  ]
 },
 {
  "outcome": "the king will sieze the lan[d of his enemy and?] the weapon of the king will lay the land low",
  "polarity": "favorable",
  "tags": [
   "king",
   "land",
   "enemy/war"
  ]
 },
 {
  "outcome": "[...]",
  "polarity": "broken/missing",
  "tags": []
 }
]

PASS 1 — polarity:
                    total  broken  unfav  neut  fav  % bad  % good
Ug. animal birth      34       6     17     4    7     61      25
Ug. lunar              8       0      3     3    2     38      25
Bab. foetus           37       0     19     7   11     51      30
Bab. izbu (human)     24       0     12     7    5     50      21
Bab. eclipse          54       0     32    19    3     59       6


In [12]:
import plotly.graph_objects as go
cc = ["unfav", "neut", "fav"]
palette = {"unfav": "#e07a5f", "neut": "#cbd5e1", "fav": "#7fb285"}
read = tbl1[cc].sum(axis=1); frac = tbl1[cc].div(read, axis=0) * 100
ylab = [f"{n}  (n={tbl1.loc[n,'total']}" +
        (f", {tbl1.loc[n,'broken']} broken)" if tbl1.loc[n,'broken'] else ")") for n in tbl1.index]
fig = go.Figure()
for k, full in [("unfav","unfavorable"), ("neut","neutral"), ("fav","favorable")]:
    fig.add_trace(go.Bar(name=full, y=ylab, x=frac[k].round(0), orientation="h",
                         marker_color=palette[k],
                         text=[f"{v:.0f}%" if v >= 7 else "" for v in frac[k]], textposition="inside"))
fig.update_layout(barmode="stack", title="Pass 1 — mood of the readable outcomes (keyword)",
                  xaxis_title="% of readable outcomes", height=340, width=950, bargap=0.35,
                  legend=dict(orientation="h", y=1.12), margin=dict(l=240, r=30, t=80, b=40),
                  paper_bgcolor="white", plot_bgcolor="white")
fig.show()

This first pass is **rough**. Two failure modes to watch: **perspective** (*"the land of the enemy will perish"* is good news, but the rule sees "perish") and **tag conflation** (is "perish" death, war, or natural disaster?). §9 hands exactly these to an LLM.

## 9. Pass 2 — let an LLM refine the annotations

Pass 1 is a baseline a machine makes in milliseconds. **Pass 2** asks an LLM to *review and correct* it — fixing perspective, splitting the death / disaster / war senses, and re-judging which outcomes are too **broken** to read. We ship a pre-baked refinement (`annotations_pass2.json`) so the comparison works offline.

> **Try it live:** copy the prompt below together with the raw `data/omens/annotations_pass1.json` into your own LLM (ChatGPT, Claude, Gemini…) and compare its JSON to ours. Where do you — and it — disagree with the keyword pass?

In [13]:
PROMPT = """You are refining a keyword-generated annotation of ancient Mesopotamian / Ugaritic
omen OUTCOMES (apodoses). For each outcome return JSON {"outcome", "polarity", "tags"}.

polarity — one of favorable / unfavorable / neutral / "broken/missing", judged from the
  perspective of the omen's CLIENT (the king / "our" side):
    • "the king will destroy his enemy"   -> favorable
    • "the enemy will devour the land"     -> unfavorable
    • "the land of the enemy will perish"  -> favorable   (it is the ENEMY's land!)
    • use "broken/missing" ONLY if the outcome is too fragmentary to read.

tags — zero or more of: king, land, enemy/war, famine/harvest, dynasty/succession, people,
  cattle/livestock, death/disease, disaster.  Multi-label. Keep death/disease (mortality,
  plague) separate from disaster (deluge, flood, drought, cataclysm); war belongs to enemy/war.

Fix the keyword pass's mistakes. Return only the JSON array."""
print(PROMPT)
print("\n→ paste this together with the raw file: data/omens/annotations_pass1.json")

You are refining a keyword-generated annotation of ancient Mesopotamian / Ugaritic
omen OUTCOMES (apodoses). For each outcome return JSON {"outcome", "polarity", "tags"}.

polarity — one of favorable / unfavorable / neutral / "broken/missing", judged from the
  perspective of the omen's CLIENT (the king / "our" side):
    • "the king will destroy his enemy"   -> favorable
    • "the enemy will devour the land"     -> unfavorable
    • "the land of the enemy will perish"  -> favorable   (it is the ENEMY's land!)
    • use "broken/missing" ONLY if the outcome is too fragmentary to read.

tags — zero or more of: king, land, enemy/war, famine/harvest, dynasty/succession, people,
  cattle/livestock, death/disease, disaster.  Multi-label. Keep death/disease (mortality,
  plague) separate from disaster (deluge, flood, drought, cataclysm); war belongs to enemy/war.

Fix the keyword pass's mistakes. Return only the JSON array.

→ paste this together with the raw file: data/omens/annotations_pas

In [14]:
import json, pathlib
pass2 = json.loads(pathlib.Path("../data/omens/annotations_pass2.json").read_text(encoding="utf-8"))
pass2 = {k: v for k, v in pass2.items() if not k.startswith("_")}

def index(ann): return {r["outcome"]: r for recs in ann.values() for r in recs}
i1, i2 = index(pass1), index(pass2)
pol = [(o, i1[o]["polarity"], i2[o]["polarity"]) for o in i1 if i1[o]["polarity"] != i2[o]["polarity"]]
tag = [o for o in i1 if i1[o]["tags"] != i2[o]["tags"]]
print(f"the LLM pass changed POLARITY on {len(pol)} outcomes and TAGS on {len(tag)}.\n")
print("examples (polarity):")
for o, a, b in pol[:8]:
    print(f"  {a:14} → {b:14} | {o[:58]}")

the LLM pass changed POLARITY on 25 outcomes and TAGS on 25.

examples (polarity):
  unfavorable    → favorable      | the land of the enemy will perish
  neutral        → unfavorable    | the seed-grain will be affected
  favorable      → unfavorable    | the enemy will devastate and consume the land
  neutral        → broken/missing | [...] assembly
  neutral        → broken/missing | the king [...]
  neutral        → broken/missing | the word (interpretation?) [...]
  neutral        → broken/missing | the word (interpretation?) [...]; that arrives where the m
  neutral        → broken/missing | son(s) of Baʼlu? [...]


### The mood, pass 1 vs pass 2

Did refining the labels change *how ominous* each tradition looks? Each tradition shows two bars — keyword **pass 1** (top) and LLM **pass 2** (below) — as a share of its **readable** outcomes. (The denominator shifts a little where pass 2 re-flags an outcome as broken; hover for the raw counts.)

In [ ]:
import collections
import plotly.graph_objects as go

order = ["unfavorable", "neutral", "favorable"]
palette = {"unfavorable": "#e07a5f", "neutral": "#cbd5e1", "favorable": "#7fb285"}
SETS5 = [s for s in pass2 if s != "Ug. dream"]

def mood(ann, s):
    c = collections.Counter(r["polarity"] for r in ann[s])
    read = c["unfavorable"] + c["neutral"] + c["favorable"]
    return {k: (100 * c[k] / read if read else 0) for k in order}, read

rows = []                                   # one row per (set, pass)
for s in SETS5:
    f1, n1 = mood(pass1, s); f2, n2 = mood(pass2, s)
    rows.append((f"{s} · pass 1", f1, n1))
    rows.append((f"{s} · pass 2", f2, n2))
ylab = [r[0] for r in rows]

fig = go.Figure()
for k in order:
    fig.add_trace(go.Bar(
        y=ylab, x=[r[1][k] for r in rows], name=k, orientation="h", marker_color=palette[k],
        text=[f"{r[1][k]:.0f}%" if r[1][k] >= 8 else "" for r in rows], textposition="inside",
        customdata=[[r[2]] for r in rows],
        hovertemplate="%{y}<br>" + k + ": %{x:.0f}% of %{customdata[0]} readable<extra></extra>"))
fig.update_layout(barmode="stack",
    title="Mood by tradition — pass 1 (keyword) vs pass 2 (LLM-refined)",
    xaxis_title="% of readable outcomes", height=560, width=950, bargap=0.22,
    legend=dict(orientation="h", y=1.06), margin=dict(l=215, r=30, t=70, b=40),
    yaxis=dict(autorange="reversed"), paper_bgcolor="white", plot_bgcolor="white")
fig.show()

In [15]:
import numpy as np
import plotly.graph_objects as go

cats = list(TAGS)
SETS5 = [s for s in pass2 if s != "Ug. dream"]
counts = pd.DataFrame(0, index=SETS5, columns=cats, dtype=int)
denom = {}
for s in SETS5:
    read = [r for r in pass2[s] if r["polarity"] != "broken/missing"]
    denom[s] = len(read)
    for r in read:
        for t in r["tags"]:
            counts.loc[s, t] += 1
pct = counts.div(pd.Series(denom), axis=0).mul(100).round().astype(int)

ug  = pct.loc[[i for i in pct.index if i.startswith("Ug")]].mean()
bab = pct.loc[[i for i in pct.index if i.startswith("Bab")]].mean()
cos = float(np.dot(ug, bab) / (np.linalg.norm(ug) * np.linalg.norm(bab)))
print(pct.to_string()); print(f"\nUgarit vs Babylon concern cosine (refined): {cos:.2f}")

# customdata carries the raw count and the per-row denominator → shown on hover
cd = np.dstack([counts.values, np.array([[denom[s]] * len(cats) for s in SETS5])])
fig = go.Figure(go.Heatmap(
    z=pct.values, x=pct.columns.tolist(), y=pct.index.tolist(), customdata=cd,
    colorscale="YlOrRd", zmin=0, zmax=int(pct.values.max()),
    text=pct.values, texttemplate="%{text}", textfont=dict(size=11),
    hovertemplate=("<b>%{y}</b> · %{x}<br>"
                   "%{customdata[0]} of %{customdata[1]} readable outcomes  (%{z}%)<extra></extra>"),
    colorbar=dict(title="% of<br>outcomes")))
fig.update_layout(
    title="What do the omens predict about? — % of each set's outcomes per concern (pass 2, refined)",
    height=440, width=1000, margin=dict(l=165, r=40, t=60, b=150),
    xaxis=dict(side="bottom", tickangle=-30, tickfont=dict(size=12)),
    yaxis=dict(autorange="reversed", tickfont=dict(size=12)),
    paper_bgcolor="white")
fig.show()

                   king  land  enemy/war  famine/harvest  dynasty/succession  people  cattle/livestock  death/disease  disaster
Ug. animal birth     43    50         43              11                   0       0                14              7        11
Ug. lunar            17     0          0              17                   0      17                17             17         0
Bab. foetus          49    54         16              16                  16      16                 5             11         5
Bab. izbu (human)    42    46          8               4                  21      33                 0              4         4
Bab. eclipse         31    43         31              19                  11      22                 4             24         4

Ugarit vs Babylon concern cosine (refined): 0.90


**What the LLM pass bought us.** It flipped the **perspective** cases (the enemy's misfortune is *our* good news), recognized that the **dream** tablet's interpretations are all lost (→ broken), and untangled **death/disease** from natural **disaster** and **war**. Crucially, the *headline patterns survive the cleanup*: divination is still overwhelmingly a **warning system**, and **king + land** still dominate every tradition — the Ugarit↔Babylon concern-profiles even tighten from **0.89 → 0.90** cosine. The refinement sharpens the edges; it does not overturn the story.

But pass 2 is **not ground truth** either — several calls are debatable (is *"the seed-grain will be affected"* bad or neutral?). That's the standing lesson: keyword baseline → LLM refinement → **human** adjudication. → §10.

## 10. Where LLMs help — and where they are dangerous
**Help (you just saw it):** §9 used an LLM to *refine* a keyword annotation — fixing perspective, splitting the death/disaster/war senses, re-judging broken outcomes — and the headline patterns held. LLMs also extract structure from raw text, validate JSON, and explain results in plain language.

**Danger:** they **invent** what the tablet doesn't say (the `[...]` gaps), **smooth over** ambiguity, and — worst — **confidently** mislabel where they should hesitate. Pass 2 is *not* ground truth: some of its calls are debatable, so a human still adjudicates.

> **The loop:** keyword baseline → LLM refinement → **human** check. Paste an omen text (or `annotations_pass1.json`) into an LLM and watch both the help and the overreach for yourself.